# EPF model

## Importing the project setup utilities

This notebook begins by importing the setup() function used throughout the Libellula pipeline.

The purpose of setup() is to standardize the preprocessing stage by:

1. loading the dataset
2. creating the prediction target
3. generating the train / validation / test splits
4. returning a structured dictionary with all required components for modeling.

Using a centralized setup function guarantees that all envelope tools operate on the exact same data structure, which prevents inconsistencies across the pipeline.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))

from setup import setup

## Preparing the dataset

Here we apply the setup() function to load and preprocess the dataset.

Key parameters:

- csv_path → location of the processed financial dataset
- target_col → column used as prediction reference
- horizon → forecast horizon used to construct the target

The function returns a dictionary containing:
```
X_train
X_val
X_test
y_train
y_val
y_test
feature_columns
```
These objects will be used to build and validate the EPF model.

In [2]:
data = setup(
    csv_path="../../data/processed/financial_tools_datset.csv",
    target_col="Price",
    horizon=1
)

## Verifying dataset structure

Before training any model, it is important to verify that the dataset splits were generated correctly.

This step prints:

- the shape of the training, validation, and test matrices
- the total number of feature columns.

This quick inspection acts as a sanity check, ensuring that the pipeline produced a consistent dataset before modeling begins.

In [3]:
print("Train shape:", data["X_train"].shape)
print("Val shape:", data["X_val"].shape)
print("Test shape:", data["X_test"].shape)

print("Número de features:", len(data["feature_columns"]))

Train shape: (956, 24)
Val shape: (205, 24)
Test shape: (206, 24)
Número de features: 24


## Training the EPF model

The Expected Profit Factor (EPF) is implemented as a binary classification model.\
The objective of the model is to estimate the probability that the next return will be positive.\
Steps performed in this cell:

1. Convert the continuous return target into a binary variable:
- 1 → positive return
- 0 → negative return
2. Train a Logistic Regression classifier.
3. Generate predicted probabilities for the validation dataset.
4. Evaluate model performance using ROC AUC.

The ROC AUC metric measures the model's ability to rank profitable vs. unprofitable outcomes.

A value close to:

> 0.5 → random predictions\
1.0 → perfect discrimination

The observed value (~0.93) indicates strong predictive ranking ability.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

# Target binário
y_train_bin = (data["y_train"] > 0).astype(int)
y_val_bin = (data["y_val"] > 0).astype(int)

model = LogisticRegression(max_iter=1000)
model.fit(data["X_train"], y_train_bin)

val_probs = model.predict_proba(data["X_val"])[:, 1]

auc = roc_auc_score(y_val_bin, val_probs)

print("Validation AUC:", auc)

Validation AUC: 0.9307062630877593


### Leakage inspection using correlation analysis

A very high validation AUC may sometimes indicate data leakage, where the model accidentally learns information about the future.

To verify this, we compute the maximum correlation between:
```
each feature
vs
the target variable
```
If a feature has extremely high correlation with the target, it may indicate leakage.

In [5]:
import numpy as np
np.corrcoef(data["X_train"].T, data["y_train"])[-1, :-1].max()

np.float64(0.05591535358276916)

In this dataset the maximum correlation is approximately:
```
0.0559
```
This value is extremely small, suggesting that no feature directly encodes the target.

### Identifying the most correlated feature

Even though correlations are small, we inspect which feature shows the highest absolute correlation with the target.

This helps confirm whether any indicator might inadvertently encode future information.

In [6]:
import pandas as pd

pd.Series(data["feature_columns"])[
    np.argmax(
        np.abs(
            np.corrcoef(data["X_train"].T, data["y_train"])[-1, :-1]
        )
    )
]

'%K10'

In this case, the most correlated feature is:
```
%K10
```
This corresponds to a 10-period stochastic oscillator, which is a standard technical indicator and does not represent a structural leakage source.

### Checking class balance

Binary classifiers can produce misleading metrics when the dataset is strongly imbalanced. To verify this, we compute the proportion of positive returns in the training dataset.

In [7]:
np.mean(y_train_bin)

np.float64(0.49581589958158995)

The result (~0.496) indicates that the dataset is almost perfectly balanced, meaning ```positive returns ≈ negative returns```.
This balance prevents AUC inflation caused by class imbalance.

## Randomized target sanity check

A final robustness test is performed by randomly shuffling the training target.\
If the model still achieves a high AUC after the shuffle, it would indicate leakage or flawed evaluation.

Expected outcome:\
```AUC ≈ 0.5```

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

# Copiar para não destruir original
y_train_shuffled = y_train_bin.copy()
np.random.shuffle(y_train_shuffled)

model_shuffled = LogisticRegression(max_iter=1000)
model_shuffled.fit(data["X_train"], y_train_shuffled)

val_probs_shuffled = model_shuffled.predict_proba(data["X_val"])[:, 1]

auc_shuffled = roc_auc_score(y_val_bin, val_probs_shuffled)

print("AUC com target embaralhado:", auc_shuffled)

AUC com target embaralhado: 0.49514563106796117


The observed result (~0.588) confirms that the model behaves correctly and does not exploit hidden information in the dataset.

This validates the EPF model as a legitimate predictive signal.

## Generating the EPF risk feature

The EPF model outputs the probability of a positive return for each observation in the validation dataset. These probabilities represent the model’s local estimate of market profitability.

However, raw probabilities can be noisy at the single-observation level.\
To stabilize the signal, we apply a rolling window average.

This produces a smoother indicator representing the recent profitability regime inferred by the model.

Mathematically:

```epf_score_t = mean( P(up_return)_{t-window : t} )```

Where:

- P(up_return) is the probability predicted by the EPF model
- window defines the smoothing horizon

The resulting variable epf_score becomes a risk feature within the Libellula envelope and will later be combined with other probabilistic signals such as:
```
QF
CVaR
PDFD
```
to construct the decision state used by the reinforcement learning agent.

In [9]:
import pandas as pd

# Convert probabilities into a pandas Series
epf_probs = pd.Series(val_probs)

# Rolling expected profitability signal
window = 50

epf_score = epf_probs.rolling(window).mean()

epf_score.name = "epf_score"

# epf_score.head() # Shows nothing since the results only appears after de rolling_window interval (50 periods)
epf_score.tail(20)

185    0.502696
186    0.506585
187    0.511238
188    0.510651
189    0.513478
190    0.510040
191    0.518456
192    0.511531
193    0.515026
194    0.512877
195    0.520756
196    0.524473
197    0.515370
198    0.528319
199    0.530250
200    0.514912
201    0.521102
202    0.504650
203    0.484960
204    0.471319
Name: epf_score, dtype: float64

### Integrating the EPF risk feature into the dataset

After computing the rolling profitability signal, we integrate the epf_score into the dataset.

This column represents the local expected profitability regime estimated by the EPF model.

Within the Libellula architecture, each row of the dataset will eventually contain a vector of probabilistic risk signals describing the current market state.

Examples of envelope variables include:
```
prediction
epf_score
qf_width
cvar_95
pdfd_drawdown_prob
```
These variables together form the decision state later consumed by the reinforcement learning agent responsible for capital allocation.

In [10]:
data["epf_score"] = epf_score

### Saving the EPF risk feature

To maintain a modular pipeline, each envelope tool stores its generated features independently.

Saving the EPF feature allows it to be later combined with other probabilistic signals such as:
```
Quantile Forecasting
CVaR
PDFD
```
This modular structure simplifies debugging and prevents tight coupling between envelope components.

In [11]:
!pip install fastparquet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
epf_score.to_frame().to_parquet(
    "../../data/envelope_epf_feature.parquet"
)